# 05 - Segmentation Optimization

## Objective

This notebook investigates whether a loss function designed to
penalize false negatives more strongly can improve brain tumor
segmentation, particularly for gliomas.

## Experimental Hypothesis

The dataset exhibits substantial foreground-background imbalance,
with tumor regions occupying a small proportion of MRI pixels.

A Tversky-based loss may help reduce missed tumor regions by
assigning a higher penalty to false negatives.

## Experimental Design

The experiment compares:

- **Reference model:** Augmented U-Net with BCE + Dice loss.
- **Candidate model:** Augmented U-Net with BCE + Tversky loss.

The architecture, preprocessing, dataset partitions, data
augmentation, optimizer, and training configuration are kept
unchanged.

The candidate model is trained from scratch.

## Evaluation Strategy

Model comparison will be performed on the validation set,
with particular attention to:

- Overall Dice and IoU.
- Glioma Dice and IoU.
- Severe glioma failures (Dice < 0.2).
- False-negative and false-positive behavior.

The test set will not be used for hyperparameter selection.

## 5.2 - Tversky-Based Loss

The Tversky coefficient generalizes Dice by assigning different
weights to false positives and false negatives.

We use:

- Alpha = 0.3 (false-positive penalty)
- Beta = 0.7 (false-negative penalty)

The candidate loss combines binary cross-entropy and soft
Tversky loss.

These coefficients are experimental choices, not values
optimized on the test set.

In [ ]:
import tensorflow as tf

ALPHA = 0.3
BETA = 0.7
SMOOTH = 1e-6


def tversky_coefficient(y_true, y_pred):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    axes = [1, 2, 3]

    tp = tf.reduce_sum(y_true * y_pred, axis=axes)

    fp = tf.reduce_sum(
        (1.0 - y_true) * y_pred,
        axis=axes
    )

    fn = tf.reduce_sum(
        y_true * (1.0 - y_pred),
        axis=axes
    )

    tversky = (
        (tp + SMOOTH)
        / (tp + ALPHA * fp + BETA * fn + SMOOTH)
    )

    return tf.reduce_mean(tversky)


def bce_tversky_loss(y_true, y_pred):

    bce = tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(
            y_true, y_pred
        )
    )

    tversky_loss = 1.0 - tversky_coefficient(
        y_true, y_pred
    )

    return bce + tversky_loss


print("BCE + Tversky loss defined.")

BCE + Tversky loss defined.


In [ ]:
y_true = tf.constant(
    [[[[1.0], [1.0]],
      [[0.0], [0.0]]]]
)

perfect_pred = tf.identity(y_true)

missed_tumor = tf.zeros_like(y_true)

perfect_score = tversky_coefficient(
    y_true, perfect_pred
)

missed_score = tversky_coefficient(
    y_true, missed_tumor
)

print("Perfect prediction:", round(float(perfect_score), 4))
print("Missed tumor:", round(float(missed_score), 4))

assert float(perfect_score) > 0.999
assert float(missed_score) < 0.001

print("Tversky loss validation passed.")

Perfect prediction: 1.0
Missed tumor: 0.0
Tversky loss validation passed.


## 5.3 - Data Pipeline Reconstruction

The original train, validation, and test partitions are reused
from the data preparation artifacts.

MRI images are resized to 256 × 256 and normalized to [0, 1].
Segmentation masks are resized using nearest-neighbor interpolation
and binarized.

The training dataset uses synchronized horizontal flips and
90-degree rotations, consistent with the previous experiment.

Validation and test datasets remain unaugmented.

In [ ]:
from pathlib import Path
import urllib.request
import json
import pandas as pd
import numpy as np
import tensorflow as tf

REPO = "nouraastafofana-droid/brain-tumor-mri-segmentation"
BRANCH = "main"

ARTIFACT_DIR = Path("/content/artifacts/data_preparation")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    f"https://raw.githubusercontent.com/"
    f"{REPO}/{BRANCH}/artifacts/data_preparation"
)

for filename in [
    "train.csv",
    "val.csv",
    "test.csv",
    "preprocessing_config.json",
    "split_summary.json"
]:
    urllib.request.urlretrieve(
        f"{BASE_URL}/{filename}",
        ARTIFACT_DIR / filename
    )

train_df = pd.read_csv(ARTIFACT_DIR / "train.csv")
val_df = pd.read_csv(ARTIFACT_DIR / "val.csv")
test_df = pd.read_csv(ARTIFACT_DIR / "test.csv")

with open(ARTIFACT_DIR / "preprocessing_config.json") as f:
    config = json.load(f)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 3345
Validation: 588
Test: 853


In [ ]:
from google.colab import userdata
import os

DATA_ROOT = Path("/content/data")
BRISC_ROOT = DATA_ROOT / "brisc2025"

if not (BRISC_ROOT / "manifest.csv").exists():

    os.environ["KAGGLE_API_TOKEN"] = userdata.get(
        "KAGGLE_API_TOKEN"
    )

    DATA_ROOT.mkdir(parents=True, exist_ok=True)

    !kaggle datasets download -d briscdataset/brisc2025 -p /content/data --unzip

assert (BRISC_ROOT / "manifest.csv").exists()

print("BRISC dataset available.")

Dataset URL: https://www.kaggle.com/datasets/briscdataset/brisc2025
License(s): Attribution 4.0 International (CC BY 4.0)
100% 250M/250M [00:15<00:00, 16.5MB/s]

BRISC dataset available.


In [ ]:
IMAGE_SIZE = tuple(config["image_size"])
BATCH_SIZE = config["batch_size"]
RANDOM_STATE = config["random_state"]

AUTOTUNE = tf.data.AUTOTUNE


def preprocess_pair(image_path, mask_path):

    image = tf.io.read_file(image_path)
    image = tf.io.decode_image(
        image,
        channels=1,
        expand_animations=False
    )
    image.set_shape([None, None, 1])

    mask = tf.io.read_file(mask_path)
    mask = tf.io.decode_png(mask, channels=1)

    image = tf.image.resize(
        image,
        IMAGE_SIZE,
        method=config["image_interpolation"]
    )

    mask = tf.image.resize(
        mask,
        IMAGE_SIZE,
        method=config["mask_interpolation"]
    )

    image = tf.cast(image, tf.float32) / 255.0
    mask = tf.cast(
        mask >= config["mask_threshold"],
        tf.float32
    )

    return image, mask


def build_dataset(df, training=False):

    image_paths = [
        str(BRISC_ROOT / p.replace("\\", "/"))
        for p in df["image_relative_path"]
    ]

    mask_paths = [
        str(BRISC_ROOT / p.replace("\\", "/"))
        for p in df["mask_relative_path"]
    ]

    assert all(Path(p).is_file() for p in image_paths)
    assert all(Path(p).is_file() for p in mask_paths)

    ds = tf.data.Dataset.from_tensor_slices(
        (image_paths, mask_paths)
    )

    if training:
        ds = ds.shuffle(
            len(df),
            seed=RANDOM_STATE,
            reshuffle_each_iteration=True
        )

    ds = ds.map(
        preprocess_pair,
        num_parallel_calls=AUTOTUNE
    )

    return ds


def augment_pair(image, mask):

    flip = tf.random.uniform(()) < 0.5

    image = tf.cond(
        flip,
        lambda: tf.image.flip_left_right(image),
        lambda: image
    )

    mask = tf.cond(
        flip,
        lambda: tf.image.flip_left_right(mask),
        lambda: mask
    )

    k = tf.random.uniform(
        (),
        minval=0,
        maxval=4,
        dtype=tf.int32
    )

    image = tf.image.rot90(image, k)
    mask = tf.image.rot90(mask, k)

    return image, mask


train_aug_ds = (
    build_dataset(train_df, training=True)
    .map(augment_pair, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    build_dataset(val_df)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("Training and validation datasets ready.")

Training and validation datasets ready.


In [ ]:
for name, ds in [
    ("Train augmented", train_aug_ds),
    ("Validation", val_ds)
]:
    images, masks = next(iter(ds))

    print(f"\n{name}")
    print("Images:", images.shape)
    print("Masks:", masks.shape)
    print(
        "Mask values:",
        tf.unique(tf.reshape(masks, [-1]))[0].numpy()
    )

    assert images.shape[1:] == (256, 256, 1)
    assert masks.shape[1:] == (256, 256, 1)
    assert tf.reduce_all(
        tf.logical_or(masks == 0, masks == 1)
    )

print("\nData pipeline validated.")


Train augmented
Images: (16, 256, 256, 1)
Masks: (16, 256, 256, 1)
Mask values: [0. 1.]

Validation
Images: (16, 256, 256, 1)
Masks: (16, 256, 256, 1)
Mask values: [0. 1.]

Data pipeline validated.


## 5.4 — U-Net Architecture and Model Compilation

The same U-Net architecture used in the baseline and
augmentation experiments is retained.

Architecture:
- Encoder filters: 32, 64, 128, 256
- Bottleneck filters: 512
- Decoder with transposed convolutions and skip connections
- Sigmoid output for binary segmentation

The model is initialized from scratch.

The main experimental change is the loss function:
BCE + Tversky instead of BCE + Dice.

The Adam optimizer, learning rate, and evaluation metrics
remain unchanged.

In [ ]:
from tensorflow.keras import layers, models


def conv_block(x, filters):
    x = layers.Conv2D(
        filters, 3, activation="relu", padding="same"
    )(x)
    x = layers.Conv2D(
        filters, 3, activation="relu", padding="same"
    )(x)
    return x


def encoder_block(x, filters):
    skip = conv_block(x, filters)
    pooled = layers.MaxPooling2D((2, 2))(skip)
    return skip, pooled


def decoder_block(x, skip, filters):
    x = layers.Conv2DTranspose(
        filters, (2, 2), strides=2, padding="same"
    )(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x


def build_unet(input_shape=(256, 256, 1)):
    inputs = layers.Input(shape=input_shape)

    s1, p1 = encoder_block(inputs, 32)
    s2, p2 = encoder_block(p1, 64)
    s3, p3 = encoder_block(p2, 128)
    s4, p4 = encoder_block(p3, 256)

    bottleneck = conv_block(p4, 512)

    d1 = decoder_block(bottleneck, s4, 256)
    d2 = decoder_block(d1, s3, 128)
    d3 = decoder_block(d2, s2, 64)
    d4 = decoder_block(d3, s1, 32)

    outputs = layers.Conv2D(
        1, (1, 1), activation="sigmoid"
    )(d4)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="unet_bce_tversky"
    )


model_tversky = build_unet()

print("Total parameters:", model_tversky.count_params())

assert model_tversky.count_params() == 7_759_521

Total parameters: 7759521


In [ ]:
def dice_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=[1, 2, 3]
    )

    denominator = (
        tf.reduce_sum(y_true, axis=[1, 2, 3])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3])
    )

    dice = (2.0 * intersection + 1e-6) / (
        denominator + 1e-6
    )

    return tf.reduce_mean(dice)


def iou_coefficient(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred >= 0.5, tf.float32)

    intersection = tf.reduce_sum(
        y_true * y_pred, axis=[1, 2, 3]
    )

    union = (
        tf.reduce_sum(y_true, axis=[1, 2, 3])
        + tf.reduce_sum(y_pred, axis=[1, 2, 3])
        - intersection
    )

    iou = (intersection + 1e-6) / (union + 1e-6)

    return tf.reduce_mean(iou)

In [ ]:
model_tversky.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=bce_tversky_loss,
    metrics=[
        dice_coefficient,
        iou_coefficient
    ]
)

print("Model compiled successfully.")
print("Loss: BCE + Tversky")
print("Alpha:", ALPHA)
print("Beta:", BETA)
print("Learning rate: 0.0001")

Model compiled successfully.
Loss: BCE + Tversky
Alpha: 0.3
Beta: 0.7
Learning rate: 0.0001


In [ ]:
sample_images, sample_masks = next(iter(train_aug_ds))

sample_predictions = model_tversky(
    sample_images[:2],
    training=False
)

print("Input shape:", sample_images[:2].shape)
print("Output shape:", sample_predictions.shape)

assert sample_predictions.shape == (2, 256, 256, 1)

print("U-Net architecture validated.")

Input shape: (2, 256, 256, 1)
Output shape: (2, 256, 256, 1)
U-Net architecture validated.


## 5.5 - Model Training

The U-Net is trained from scratch using BCE + Tversky loss.

To ensure a controlled comparison with the augmented U-Net
experiment, the training configuration remains unchanged:

- Adam optimizer with an initial learning rate of 1e-4.
- Batch size of 16.
- Maximum of 30 epochs.
- Early stopping based on validation loss (patience = 7).
- Learning rate reduction based on validation loss
  (factor = 0.5, patience = 3).
- Best-model checkpoint selected using validation loss.

The training dataset uses synchronized data augmentation.
The validation dataset remains unchanged.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path(
    "/content/drive/MyDrive/brain-tumor-mri-segmentation"
)

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULTS_DIR = PROJECT_DIR / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive directories ready.")

Mounted at /content/drive
Google Drive directories ready.


In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint,
    CSVLogger
)

checkpoint_path = (
    CHECKPOINT_DIR / "unet_tversky_best.keras"
)

training_log_path = (
    RESULTS_DIR / "unet_tversky_training_log.csv"
)

callbacks = [
    ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),

    EarlyStopping(
        monitor="val_loss",
        patience=7,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        mode="min",
        min_lr=1e-7,
        verbose=1
    ),

    CSVLogger(
        filename=str(training_log_path)
    )
]

print("Callbacks configured.")
print("Checkpoint:", checkpoint_path)
print("Training log:", training_log_path)

Callbacks configured.
Checkpoint: /content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints/unet_tversky_best.keras
Training log: /content/drive/MyDrive/brain-tumor-mri-segmentation/results/unet_tversky_training_log.csv


In [ ]:
history_tversky = model_tversky.fit(
    train_aug_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step - dice_coefficient: 0.7828 - iou_coefficient: 0.6914 - loss: 0.2345
Epoch 1: val_loss did not improve from 0.20755
210/210 ━━━━━━━━━━━━━━━━━━━━ 72s 345ms/step - dice_coefficient: 0.7816 - iou_coefficient: 0.6909 - loss: 0.2350 - val_dice_coefficient: 0.8045 - val_iou_coefficient: 0.7181 - val_loss: 0.2114 - learning_rate: 1.0000e-04
Epoch 2/30
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - dice_coefficient: 0.7857 - iou_coefficient: 0.6961 - loss: 0.2303
Epoch 2: val_loss did not improve from 0.20755
210/210 ━━━━━━━━━━━━━━━━━━━━ 71s 338ms/step - dice_coefficient: 0.7924 - iou_coefficient: 0.7027 - loss: 0.2225 - val_dice_coefficient: 0.8135 - val_iou_coefficient: 0.7330 - val_loss: 0.2084 - learning_rate: 1.0000e-04
Epoch 3/30
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 330ms/step - dice_coefficient: 0.7981 - iou_coefficient: 0.7092 - loss: 0.2175
Epoch 3: val_loss did not improve from 0.20755

Epoch 3: ReduceLROnPlateau reducing learning rate to

In [ ]:
import pandas as pd

training_history = pd.read_csv(training_log_path)

best_epoch_idx = training_history["val_loss"].idxmin()
best_epoch = training_history.loc[best_epoch_idx]

print("Best epoch:", int(best_epoch["epoch"]) + 1)
print("Best validation loss:", round(best_epoch["val_loss"], 4))
print(
    "Validation Dice:",
    round(best_epoch["val_dice_coefficient"], 4)
)
print(
    "Validation IoU:",
    round(best_epoch["val_iou_coefficient"], 4)
)

print("\nCheckpoint saved:", checkpoint_path.exists())
print("Training log saved:", training_log_path.exists())

Best epoch: 26
Best validation loss: 0.1594
Validation Dice: 0.8509
Validation IoU: 0.7708

Checkpoint saved: True
Training log saved: True


## 5.6 - Detailed Validation Evaluation

The best BCE + Tversky model is evaluated on the unchanged
validation set.

Per-image metrics include:

- Dice coefficient
- Intersection over Union (IoU)
- Pixel-level precision
- Pixel-level recall

The analysis focuses particularly on glioma segmentation
and severe failures (Dice < 0.2).

The evaluation uses a fixed prediction threshold of 0.5.
No test images are used.

In [ ]:
import numpy as np
import pandas as pd

validation_results = val_df.copy().reset_index(drop=True)

all_metrics = []

for images, masks in val_ds:

    predictions = model_tversky.predict(
        images,
        verbose=0
    )

    y_true = masks.numpy() >= 0.5
    y_pred = predictions >= 0.5

    for true_mask, pred_mask in zip(y_true, y_pred):

        true_mask = true_mask.astype(bool)
        pred_mask = pred_mask.astype(bool)

        tp = np.logical_and(
            true_mask, pred_mask
        ).sum()

        fp = np.logical_and(
            ~true_mask, pred_mask
        ).sum()

        fn = np.logical_and(
            true_mask, ~pred_mask
        ).sum()

        dice = (
            (2 * tp + 1e-6)
            / (2 * tp + fp + fn + 1e-6)
        )

        iou = (
            (tp + 1e-6)
            / (tp + fp + fn + 1e-6)
        )

        precision = (
            (tp + 1e-6)
            / (tp + fp + 1e-6)
        )

        recall = (
            (tp + 1e-6)
            / (tp + fn + 1e-6)
        )

        all_metrics.append({
            "dice": float(dice),
            "iou": float(iou),
            "precision": float(precision),
            "recall": float(recall)
        })

assert len(all_metrics) == len(validation_results)

validation_results = pd.concat(
    [
        validation_results,
        pd.DataFrame(all_metrics)
    ],
    axis=1
)

print("Images evaluated:", len(validation_results))

print("\nOverall validation metrics:")
print(
    validation_results[
        ["dice", "iou", "precision", "recall"]
    ].mean().round(4)
)

Images evaluated: 588

Overall validation metrics:
dice         0.8510
iou          0.7709
precision    0.8432
recall       0.8949
dtype: float64


In [ ]:
print("Available columns:")
print(validation_results.columns.tolist())

Available columns:
['image_relative_path', 'mask_relative_path', 'tumor_label', 'plane_label', 'image_sha256', 'mask_sha256', 'dice', 'iou', 'precision', 'recall']


## 5.7 - Performance by Tumor Type

The model is evaluated separately for glioma, meningioma,
and pituitary tumor images.

Particular attention is given to glioma segmentation,
which showed the weakest performance in previous experiments.

Severe segmentation failures are defined as images
with a Dice coefficient below 0.2.

In [ ]:
# Performance by tumor type

tumor_summary = (
    validation_results
    .groupby("tumor_label")
    .agg(
        images=("dice", "count"),
        mean_dice=("dice", "mean"),
        median_dice=("dice", "median"),
        mean_iou=("iou", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean")
    )
    .round(4)
)

print("Performance by tumor type:")
display(tumor_summary)


# Severe segmentation failures

severe_failures = (
    validation_results
    .assign(severe_failure=lambda df: df["dice"] < 0.2)
    .groupby("tumor_label")
    .agg(
        total_images=("dice", "count"),
        severe_failures=("severe_failure", "sum"),
        failure_rate=("severe_failure", "mean")
    )
)

severe_failures["failure_rate"] = (
    severe_failures["failure_rate"] * 100
).round(2)

print("\nSevere segmentation failures:")
display(severe_failures)

Performance by tumor type:


,images,mean_dice,median_dice,mean_iou,mean_precision,mean_recall
tumor_label,,,,,,
glioma,160,0.7352,0.8192,0.6305,0.7400,0.7959
meningioma,211,0.9262,0.9500,0.8726,0.9098,0.9572
pituitary,217,0.8632,0.9038,0.7756,0.8546,0.9073



Severe segmentation failures:


,total_images,severe_failures,failure_rate
tumor_label,,,
glioma,160,13,8.12
meningioma,211,1,0.47
pituitary,217,1,0.46


## 5.8 - Paired Validation Comparison

This section compares the augmented U-Net and the
BCE + Tversky U-Net on the same validation images.

The analysis measures:

- Per-image Dice improvement or regression.
- Glioma cases improved by Tversky.
- Severe failures resolved or newly introduced.
- Changes in segmentation performance by tumor type.

This comparison is exploratory and does not establish
statistical significance.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

previous_results_path = (
    Path("/content/drive/MyDrive/brain-tumor-mri-segmentation")
    / "results"
    / "unet_augmented_validation_comparison.csv"
)

previous_results = pd.read_csv(previous_results_path)

# Identify each image-mask pair
merge_keys = [
    "image_relative_path",
    "mask_relative_path"
]

# Normalize path separators
for col in merge_keys:
    previous_results[col] = (
        previous_results[col]
        .astype(str)
        .str.replace("\\", "/", regex=False)
    )

    validation_results[col] = (
        validation_results[col]
        .astype(str)
        .str.replace("\\", "/", regex=False)
    )

# Verify pair uniqueness
print(
    "Duplicate pairs in previous results:",
    previous_results.duplicated(merge_keys).sum()
)

print(
    "Duplicate pairs in Tversky results:",
    validation_results.duplicated(merge_keys).sum()
)

assert not previous_results.duplicated(merge_keys).any()
assert not validation_results.duplicated(merge_keys).any()

# Keep only the previous model's Dice
previous_results = previous_results[
    merge_keys + ["dice"]
].rename(columns={
    "dice": "augmented_dice"
})

# Merge on the complete image-mask pair
comparison = validation_results.merge(
    previous_results,
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == len(validation_results)

comparison["dice_gain"] = (
    comparison["dice"] - comparison["augmented_dice"]
)

glioma_comparison = comparison[
    comparison["tumor_label"] == "glioma"
].copy()

print("\nMatched validation images:", len(comparison))
print("Matched glioma images:", len(glioma_comparison))

print("\nGlioma Dice comparison:")
print(
    glioma_comparison[
        ["augmented_dice", "dice", "dice_gain"]
    ].mean().round(4)
)

print(
    "\nGlioma images improved:",
    (glioma_comparison["dice_gain"] > 0).sum()
)

print(
    "Glioma images worsened:",
    (glioma_comparison["dice_gain"] < 0).sum()
)

previous_severe = glioma_comparison["augmented_dice"] < 0.2
current_severe = glioma_comparison["dice"] < 0.2

print("\nPrevious severe failures:", previous_severe.sum())
print("Current severe failures:", current_severe.sum())

print(
    "Severe failures resolved:",
    (previous_severe & ~current_severe).sum()
)

print(
    "New severe failures:",
    (~previous_severe & current_severe).sum()
)

Duplicate pairs in previous results: 0
Duplicate pairs in Tversky results: 0

Matched validation images: 588
Matched glioma images: 160

Glioma Dice comparison:
augmented_dice    0.7004
dice              0.7352
dice_gain         0.0348
dtype: float64

Glioma images improved: 89
Glioma images worsened: 70

Previous severe failures: 17
Current severe failures: 13
Severe failures resolved: 6
New severe failures: 2


## 5.9 - Save Validation Results

The BCE + Tversky U-Net achieved improved mean validation
performance, particularly for glioma segmentation.

Compared with the augmented BCE + Dice U-Net:

- Mean glioma Dice increased from 0.7004 to 0.7352.
- Severe glioma failures decreased from 17 to 13.
- Six previous severe failures were resolved.
- Two new severe failures appeared.
- Dice improved on 89 glioma images and worsened on 70.

The results support selecting BCE + Tversky as the current
experimental U-Net candidate.

In [ ]:
import json

results_dir = (
    Path("/content/drive/MyDrive/brain-tumor-mri-segmentation")
    / "results"
)

results_dir.mkdir(parents=True, exist_ok=True)

validation_path = (
    results_dir / "unet_tversky_validation_results.csv"
)

comparison_path = (
    results_dir / "unet_tversky_validation_comparison.csv"
)

summary_path = (
    results_dir / "unet_tversky_validation_summary.json"
)

validation_results.to_csv(validation_path, index=False)
comparison.to_csv(comparison_path, index=False)

summary = {
    "model": "unet_bce_tversky",
    "dataset": "validation",
    "validation_images": len(validation_results),
    "alpha": ALPHA,
    "beta": BETA,
    "mean_dice": float(validation_results["dice"].mean()),
    "mean_iou": float(validation_results["iou"].mean()),
    "mean_precision": float(validation_results["precision"].mean()),
    "mean_recall": float(validation_results["recall"].mean()),
    "glioma_images": len(glioma_comparison),
    "glioma_mean_dice": float(glioma_comparison["dice"].mean()),
    "glioma_augmented_mean_dice": float(
        glioma_comparison["augmented_dice"].mean()
    ),
    "glioma_images_improved": int(
        (glioma_comparison["dice_gain"] > 0).sum()
    ),
    "glioma_images_worsened": int(
        (glioma_comparison["dice_gain"] < 0).sum()
    ),
    "previous_severe_glioma_failures": int(previous_severe.sum()),
    "current_severe_glioma_failures": int(current_severe.sum()),
    "resolved_severe_failures": int(
        (previous_severe & ~current_severe).sum()
    ),
    "new_severe_failures": int(
        (~previous_severe & current_severe).sum()
    )
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Validation results saved:")
print(validation_path)
print(comparison_path)
print(summary_path)

Validation results saved:
/content/drive/MyDrive/brain-tumor-mri-segmentation/results/unet_tversky_validation_results.csv
/content/drive/MyDrive/brain-tumor-mri-segmentation/results/unet_tversky_validation_comparison.csv
/content/drive/MyDrive/brain-tumor-mri-segmentation/results/unet_tversky_validation_summary.json


## 5.10 - Transfer Learning with a Pretrained Encoder

This experiment investigates whether transfer learning improves
brain tumor segmentation compared with a standard U-Net.

A U-Net architecture with a pretrained ResNet50 encoder will
be evaluated.

Experimental configuration:

- Encoder: ResNet50 pretrained on ImageNet.
- Decoder: U-Net segmentation decoder.
- Input: 256 × 256 MRI images adapted to three channels.
- Output: Single-channel binary segmentation mask.
- Loss: BCE + Tversky (alpha = 0.3, beta = 0.7).
- Training augmentation: Synchronized horizontal flips and
  90-degree rotations.
- Evaluation: Unchanged validation partition.

The pretrained model will be compared with the previous
BCE + Tversky U-Net using the same validation images.


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

print("\nPretrained encoder available:")
print(tf.keras.applications.ResNet50)

TensorFlow version: 2.21.0
Keras version: 3.13.2
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Pretrained encoder available:
<function ResNet50 at 0x78b2e06cc400>


## 5.11 - Pretrained ResNet50 U-Net Architecture

This experiment uses ResNet50 pretrained on ImageNet as the
encoder of a U-Net-style segmentation network.

The encoder extracts hierarchical visual features, while the
decoder reconstructs a pixel-level segmentation mask using
skip connections.

Grayscale MRI images are replicated across three channels
and transformed using the ResNet50 preprocessing function.

The pretrained encoder is initially frozen to preserve
its learned representations.

The model will use the same BCE + Tversky loss and validation
dataset as the previous experiment.

This experiment changes both the architecture and the
weight initialization strategy.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Input: grayscale MRI normalized to [0, 1]
inputs = layers.Input(
    shape=(256, 256, 1),
    name="mri_input"
)

# Convert grayscale to RGB
x = layers.Concatenate(axis=-1)(
    [inputs, inputs, inputs]
)

# ResNet50 preprocessing expects pixel values in [0, 255]
x = layers.Rescaling(255.0)(x)

x = layers.Lambda(
    tf.keras.applications.resnet50.preprocess_input,
    name="resnet_preprocessing"
)(x)

# Pretrained encoder
encoder = tf.keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(256, 256, 3)
)

encoder.trainable = False

# Extract intermediate feature maps for skip connections
skip_layer_names = [
    "conv1_relu",         # 128 × 128
    "conv2_block3_out",   # 64 × 64
    "conv3_block4_out",   # 32 × 32
    "conv4_block6_out"    # 16 × 16
]

encoder_outputs = [
    encoder.get_layer(name).output
    for name in skip_layer_names
]

encoder_outputs.append(encoder.output)

feature_extractor = models.Model(
    encoder.input,
    encoder_outputs,
    name="resnet50_encoder"
)

features = feature_extractor(x, training=False)

s1, s2, s3, s4, bottleneck = features


def pretrained_decoder_block(x, skip, filters):

    x = layers.Conv2DTranspose(
        filters,
        kernel_size=2,
        strides=2,
        padding="same"
    )(x)

    x = layers.Concatenate()([x, skip])

    x = layers.Conv2D(
        filters, 3,
        activation="relu",
        padding="same"
    )(x)

    x = layers.Conv2D(
        filters, 3,
        activation="relu",
        padding="same"
    )(x)

    return x


# Decoder
d1 = pretrained_decoder_block(bottleneck, s4, 256)
d2 = pretrained_decoder_block(d1, s3, 128)
d3 = pretrained_decoder_block(d2, s2, 64)
d4 = pretrained_decoder_block(d3, s1, 32)

# Restore original image resolution: 128 -> 256
d5 = layers.Conv2DTranspose(
    32,
    kernel_size=2,
    strides=2,
    padding="same"
)(d4)

d5 = layers.Conv2D(
    32, 3,
    activation="relu",
    padding="same"
)(d5)

outputs = layers.Conv2D(
    1,
    kernel_size=1,
    activation="sigmoid",
    name="segmentation_mask"
)(d5)

model_resnet = models.Model(
    inputs=inputs,
    outputs=outputs,
    name="resnet50_unet"
)

print("Total parameters:", model_resnet.count_params())
print("Input shape:", model_resnet.input_shape)
print("Output shape:", model_resnet.output_shape)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Total parameters: 30553473
Input shape: (None, 256, 256, 1)
Output shape: (None, 256, 256, 1)


In [ ]:
assert model_resnet.input_shape == (None, 256, 256, 1)
assert model_resnet.output_shape == (None, 256, 256, 1)

print("ResNet50 U-Net architecture validated.")
print("Encoder frozen:", not encoder.trainable)

print(
    "Trainable parameters:",
    sum(
        int(tf.size(w))
        for w in model_resnet.trainable_weights
    )
)

ResNet50 U-Net architecture validated.
Encoder frozen: True
Trainable parameters: 6965761


## 5.12 - Compile the Pretrained U-Net

The ResNet50 encoder is initially frozen, and only the
segmentation decoder is trained.

Training configuration:

- Loss: BCE + Tversky (alpha = 0.3, beta = 0.7).
- Optimizer: Adam.
- Initial learning rate: 1e-4.
- Metrics: Dice coefficient and IoU at threshold 0.5.
- Training data: Augmented training partition.
- Validation data: Unchanged validation partition.

This first training phase evaluates the segmentation decoder
while preserving the pretrained encoder weights.

In [ ]:
model_resnet.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=bce_tversky_loss,
    metrics=[
        dice_coefficient,
        iou_coefficient
    ]
)

print("Model compiled successfully.")
print("Loss: BCE + Tversky")
print("Learning rate:", model_resnet.optimizer.learning_rate.numpy())
print("Encoder trainable:", encoder.trainable)
print("Trainable parameters:", sum(
    int(tf.size(w))
    for w in model_resnet.trainable_weights
))

Model compiled successfully.
Loss: BCE + Tversky
Learning rate: 1e-04
Encoder trainable: False
Trainable parameters: 6965761


In [ ]:
import numpy as np

# Retrieve one batch from the existing augmented dataset
sample_images, sample_masks = next(iter(train_aug_ds))

# Run inference without updating weights
sample_predictions = model_resnet(
    sample_images,
    training=False
)

# Compute loss
sample_loss = bce_tversky_loss(
    sample_masks,
    sample_predictions
)

print("Images shape:", sample_images.shape)
print("Masks shape:", sample_masks.shape)
print("Predictions shape:", sample_predictions.shape)

print("\nImage range:",
      float(tf.reduce_min(sample_images)),
      float(tf.reduce_max(sample_images)))

print("Mask range:",
      float(tf.reduce_min(sample_masks)),
      float(tf.reduce_max(sample_masks)))

print("Prediction range:",
      float(tf.reduce_min(sample_predictions)),
      float(tf.reduce_max(sample_predictions)))

print("\nInitial BCE + Tversky loss:",
      float(sample_loss))

# Sanity checks
assert sample_images.shape[-1] == 1
assert sample_masks.shape == sample_predictions.shape

assert bool(tf.reduce_all(tf.math.is_finite(sample_predictions)))
assert bool(tf.math.is_finite(sample_loss))

assert float(tf.reduce_min(sample_images)) >= 0.0
assert float(tf.reduce_max(sample_images)) <= 1.0

assert float(tf.reduce_min(sample_masks)) >= 0.0
assert float(tf.reduce_max(sample_masks)) <= 1.0

print("\nSanity check passed.")

Images shape: (16, 256, 256, 1)
Masks shape: (16, 256, 256, 1)
Predictions shape: (16, 256, 256, 1)

Image range: 0.0 0.9872549176216125
Mask range: 0.0 1.0
Prediction range: 0.3061637580394745 0.7348172664642334

Initial BCE + Tversky loss: 1.6653120517730713

Sanity check passed.


## 5.13 - Frozen-Encoder Training

The pretrained ResNet50 encoder remains frozen during the
first training phase.

Only the U-Net decoder is optimized using BCE + Tversky loss.

Training configuration:

- Maximum epochs: 15.
- Batch size: 16.
- Optimizer: Adam, learning rate 1e-4.
- Early stopping: patience of 5 epochs.
- Learning rate reduction: factor 0.5, patience of 3 epochs.
- Model selection: lowest validation loss.

The best model checkpoint and training history are saved
separately from previous experiments.

The encoder may be fine-tuned in a subsequent phase if
the initial training results justify it.

In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf

project_dir = Path(
    "/content/drive/MyDrive/brain-tumor-mri-segmentation"
)

checkpoint_dir = project_dir / "checkpoints"
results_dir = project_dir / "results"

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

resnet_checkpoint_path = (
    checkpoint_dir / "resnet50_unet_frozen_best.keras"
)

resnet_history_path = (
    results_dir / "resnet50_unet_frozen_training_log.csv"
)

resnet_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(resnet_checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(resnet_history_path)
    )
]

history_resnet_frozen = model_resnet.fit(
    train_aug_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=resnet_callbacks
)

print("\nFrozen-encoder training completed.")

print("Best validation loss:",
      min(history_resnet_frozen.history["val_loss"]))

print("Best epoch:",
      int(
          np.argmin(
              history_resnet_frozen.history["val_loss"]
          ) + 1
      ))

print("\nCheckpoint:", resnet_checkpoint_path)
print("Training log:", resnet_history_path)

Epoch 1/15
210/210 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - dice_coefficient: 0.1072 - iou_coefficient: 0.0781 - loss: 1.0983
Epoch 1: val_loss improved from None to 0.50622, saving model to /content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints/resnet50_unet_frozen_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints/resnet50_unet_frozen_best.keras
210/210 ━━━━━━━━━━━━━━━━━━━━ 104s 346ms/step - dice_coefficient: 0.3058 - iou_coefficient: 0.2351 - loss: 0.7844 - val_dice_coefficient: 0.5167 - val_iou_coefficient: 0.3768 - val_loss: 0.5062 - learning_rate: 1.0000e-04
Epoch 2/15
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 204ms/step - dice_coefficient: 0.6381 - iou_coefficient: 0.5163 - loss: 0.3968
Epoch 2: val_loss improved from 0.50622 to 0.35751, saving model to /content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints/resnet50_unet_frozen_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/brain-tumor-mri-segme

## 5.14 - Extended Frozen-Encoder Training

The best validation loss was obtained at epoch 15, the
maximum epoch of the initial training phase.

This suggests that the frozen-encoder model may benefit
from additional optimization.

Training is therefore continued for up to 10 additional
epochs without reinitializing model weights.

The ResNet50 encoder remains frozen. Early stopping and
learning-rate reduction are retained.

In [ ]:
# Continue from the weights restored at epoch 15.
# The existing model and optimizer are reused.

assert not encoder.trainable

previous_best_loss = min(
    history_resnet_frozen.history["val_loss"]
)

extended_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(resnet_checkpoint_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        initial_value_threshold=previous_best_loss,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        mode="min",
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode="min",
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename=str(resnet_history_path),
        append=True
    )
]

history_resnet_extended = model_resnet.fit(
    train_aug_ds,
    validation_data=val_ds,
    initial_epoch=15,
    epochs=25,
    callbacks=extended_callbacks
)

print("\nExtended training completed.")

print(
    "Best validation loss during extension:",
    min(history_resnet_extended.history["val_loss"])
)

print(
    "Best validation loss before extension:",
    previous_best_loss
)

print(
    "Best validation loss overall:",
    min(
        previous_best_loss,
        min(history_resnet_extended.history["val_loss"])
    )
)

Epoch 16/25
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - dice_coefficient: 0.8477 - iou_coefficient: 0.7639 - loss: 0.1598
Epoch 16: val_loss improved from 0.16632 to 0.15903, saving model to /content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints/resnet50_unet_frozen_best.keras

Epoch 16: finished saving model to /content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints/resnet50_unet_frozen_best.keras
210/210 ━━━━━━━━━━━━━━━━━━━━ 49s 230ms/step - dice_coefficient: 0.8474 - iou_coefficient: 0.7639 - loss: 0.1598 - val_dice_coefficient: 0.8453 - val_iou_coefficient: 0.7589 - val_loss: 0.1590 - learning_rate: 5.0000e-05
Epoch 17/25
209/210 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step - dice_coefficient: 0.8457 - iou_coefficient: 0.7594 - loss: 0.1599
Epoch 17: val_loss did not improve from 0.15903
210/210 ━━━━━━━━━━━━━━━━━━━━ 44s 210ms/step - dice_coefficient: 0.8468 - iou_coefficient: 0.7621 - loss: 0.1590 - val_dice_coefficient: 0.8479 - val_iou_coefficient: 0.7686 - val_loss: 0.16

## 5.15- Per-Image Validation Evaluation

The frozen-encoder ResNet50 U-Net achieved its lowest
validation loss at epoch 24.

The best checkpoint is evaluated on the unchanged validation
partition using a segmentation threshold of 0.5.

Dice, IoU, precision, and recall are calculated independently
for each image to ensure comparability with previous models.

Results are also summarized by tumor category, with
particular attention to severe glioma failures (Dice < 0.2).

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

# The best weights from epoch 24 were restored automatically.
resnet_validation_results = val_df.copy().reset_index(drop=True)

all_predictions = []
all_masks = []

for images, masks in val_ds:
    predictions = model_resnet(images, training=False)

    all_predictions.append(predictions.numpy())
    all_masks.append(masks.numpy())

all_predictions = np.concatenate(all_predictions, axis=0)
all_masks = np.concatenate(all_masks, axis=0)

assert len(all_predictions) == len(resnet_validation_results)

# Binary segmentation threshold
pred_binary = all_predictions >= 0.5
true_binary = all_masks >= 0.5

# Per-image confusion counts
axes = (1, 2, 3)

tp = np.sum(pred_binary & true_binary, axis=axes)
fp = np.sum(pred_binary & ~true_binary, axis=axes)
fn = np.sum(~pred_binary & true_binary, axis=axes)

smooth = 1e-6

resnet_validation_results["dice"] = (
    (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
)

resnet_validation_results["iou"] = (
    (tp + smooth) / (tp + fp + fn + smooth)
)

resnet_validation_results["precision"] = (
    (tp + smooth) / (tp + fp + smooth)
)

resnet_validation_results["recall"] = (
    (tp + smooth) / (tp + fn + smooth)
)

print("Validation images:", len(resnet_validation_results))

print("\nOverall mean metrics:")
print(
    resnet_validation_results[
        ["dice", "iou", "precision", "recall"]
    ].mean().round(4)
)

print("\nMetrics by tumor category:")
print(
    resnet_validation_results.groupby("tumor_label")
    .agg(
        images=("dice", "size"),
        mean_dice=("dice", "mean"),
        mean_iou=("iou", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean")
    )
    .round(4)
)

glioma_resnet = resnet_validation_results[
    resnet_validation_results["tumor_label"] == "glioma"
]

print(
    "\nSevere glioma failures (Dice < 0.2):",
    int((glioma_resnet["dice"] < 0.2).sum()),
    "/",
    len(glioma_resnet)
)

Validation images: 588

Overall mean metrics:
dice         0.8570
iou          0.7781
precision    0.8541
recall       0.8970
dtype: float64

Metrics by tumor category:
             images  mean_dice  mean_iou  mean_precision  mean_recall
tumor_label                                                          
glioma          160     0.7595    0.6593          0.7878       0.8013
meningioma      211     0.9278    0.8729          0.9043       0.9630
pituitary       217     0.8599    0.7736          0.8540       0.9034

Severe glioma failures (Dice < 0.2): 10 / 160


## 5.16 - Paired Validation Comparison

The frozen-encoder ResNet50 U-Net is compared with the
standard BCE + Tversky U-Net on identical validation pairs.

This analysis measures:

- The number of images with improved or worsened Dice.
- The average Dice difference for glioma images.
- Previously severe failures resolved by ResNet50.
- New severe failures introduced by ResNet50.

Severe segmentation failures are defined as Dice < 0.2.

In [ ]:
from pathlib import Path
import pandas as pd

tversky_results_path = (
    Path("/content/drive/MyDrive/brain-tumor-mri-segmentation")
    / "results"
    / "unet_tversky_validation_results.csv"
)

tversky_results = pd.read_csv(tversky_results_path)

merge_keys = [
    "image_relative_path",
    "mask_relative_path"
]

# Normalize paths
for col in merge_keys:
    tversky_results[col] = (
        tversky_results[col]
        .astype(str)
        .str.replace("\\", "/", regex=False)
    )

    resnet_validation_results[col] = (
        resnet_validation_results[col]
        .astype(str)
        .str.replace("\\", "/", regex=False)
    )

assert not tversky_results.duplicated(merge_keys).any()
assert not resnet_validation_results.duplicated(merge_keys).any()

# Prepare previous model results
tversky_results = tversky_results[
    merge_keys + ["dice"]
].rename(columns={
    "dice": "tversky_dice"
})

# Match identical image-mask pairs
resnet_comparison = resnet_validation_results.merge(
    tversky_results,
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

assert len(resnet_comparison) == 588

resnet_comparison["dice_gain"] = (
    resnet_comparison["dice"]
    - resnet_comparison["tversky_dice"]
)

glioma_comparison_resnet = resnet_comparison[
    resnet_comparison["tumor_label"] == "glioma"
].copy()

print("Matched validation images:", len(resnet_comparison))
print("Matched glioma images:", len(glioma_comparison_resnet))

print("\nGlioma Dice comparison:")
print(
    glioma_comparison_resnet[
        ["tversky_dice", "dice", "dice_gain"]
    ].mean().round(4)
)

print(
    "\nGlioma images improved:",
    int((glioma_comparison_resnet["dice_gain"] > 0).sum())
)

print(
    "Glioma images worsened:",
    int((glioma_comparison_resnet["dice_gain"] < 0).sum())
)

previous_severe = (
    glioma_comparison_resnet["tversky_dice"] < 0.2
)

current_severe = (
    glioma_comparison_resnet["dice"] < 0.2
)

print("\nPrevious severe failures:", int(previous_severe.sum()))
print("Current severe failures:", int(current_severe.sum()))

print(
    "Severe failures resolved:",
    int((previous_severe & ~current_severe).sum())
)

print(
    "New severe failures:",
    int((~previous_severe & current_severe).sum())
)

Matched validation images: 588
Matched glioma images: 160

Glioma Dice comparison:
tversky_dice    0.7352
dice            0.7595
dice_gain       0.0243
dtype: float64

Glioma images improved: 98
Glioma images worsened: 62

Previous severe failures: 13
Current severe failures: 10
Severe failures resolved: 5
New severe failures: 2


## 5.17 - Save ResNet50 Validation Results

The frozen-encoder ResNet50 U-Net achieved a mean validation
Dice of 0.8570 and a mean glioma Dice of 0.7595.

Compared with the standard BCE + Tversky U-Net:

- Mean glioma Dice improved by 0.0243.
- Dice improved on 98 of 160 glioma images.
- Dice worsened on 62 of 160 glioma images.
- Five previous severe glioma failures were resolved.
- Two new severe failures appeared.
- Severe glioma failures decreased from 13 to 10.

The results support selecting ResNet50 U-Net as the current
best-performing candidate on the validation partition.

In [ ]:
import json
from pathlib import Path

results_dir = (
    Path("/content/drive/MyDrive/brain-tumor-mri-segmentation")
    / "results"
)

results_dir.mkdir(parents=True, exist_ok=True)

validation_path = (
    results_dir / "resnet50_unet_validation_results.csv"
)

comparison_path = (
    results_dir / "resnet50_unet_validation_comparison.csv"
)

summary_path = (
    results_dir / "resnet50_unet_validation_summary.json"
)

resnet_validation_results.to_csv(validation_path, index=False)
resnet_comparison.to_csv(comparison_path, index=False)

summary = {
    "model": "resnet50_unet_frozen",
    "pretrained_encoder": "ResNet50_ImageNet",
    "encoder_frozen": True,
    "best_epoch": 24,
    "best_validation_loss": 0.152308851480484,
    "validation_images": len(resnet_validation_results),
    "mean_dice": float(resnet_validation_results["dice"].mean()),
    "mean_iou": float(resnet_validation_results["iou"].mean()),
    "mean_precision": float(
        resnet_validation_results["precision"].mean()
    ),
    "mean_recall": float(
        resnet_validation_results["recall"].mean()
    ),
    "glioma_images": len(glioma_comparison_resnet),
    "glioma_mean_dice": float(
        glioma_comparison_resnet["dice"].mean()
    ),
    "glioma_tversky_mean_dice": float(
        glioma_comparison_resnet["tversky_dice"].mean()
    ),
    "glioma_images_improved": int(
        (glioma_comparison_resnet["dice_gain"] > 0).sum()
    ),
    "glioma_images_worsened": int(
        (glioma_comparison_resnet["dice_gain"] < 0).sum()
    ),
    "previous_severe_glioma_failures": int(
        previous_severe.sum()
    ),
    "current_severe_glioma_failures": int(
        current_severe.sum()
    ),
    "resolved_severe_failures": int(
        (previous_severe & ~current_severe).sum()
    ),
    "new_severe_failures": int(
        (~previous_severe & current_severe).sum()
    )
}

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved:")
print(validation_path)
print(comparison_path)
print(summary_path)

Saved:
/content/drive/MyDrive/brain-tumor-mri-segmentation/results/resnet50_unet_validation_results.csv
/content/drive/MyDrive/brain-tumor-mri-segmentation/results/resnet50_unet_validation_comparison.csv
/content/drive/MyDrive/brain-tumor-mri-segmentation/results/resnet50_unet_validation_summary.json


## 5.18 - Comparative Model Summary

Four segmentation configurations were evaluated on the
same validation partition:

1. Baseline U-Net with BCE + Dice loss.
2. Augmented U-Net with BCE + Dice loss.
3. Augmented U-Net with BCE + Tversky loss.
4. ResNet50 U-Net with a frozen ImageNet-pretrained encoder
   and BCE + Tversky loss.

The ResNet50 U-Net achieved the highest mean validation
Dice, including the highest mean Dice for glioma images.

Severe glioma segmentation failures decreased from 22/160
for the baseline to 10/160 for ResNet50 U-Net.


In [ ]:
import pandas as pd
from pathlib import Path

model_summary = pd.DataFrame([
    {
        "model": "Baseline U-Net",
        "mean_dice": 0.8353,
        "glioma_mean_dice": 0.6830,
        "severe_glioma_failures": 22
    },
    {
        "model": "Augmented U-Net",
        "mean_dice": 0.8417,
        "glioma_mean_dice": 0.7004,
        "severe_glioma_failures": 17
    },
    {
        "model": "U-Net + Tversky",
        "mean_dice": 0.8510,
        "glioma_mean_dice": 0.7352,
        "severe_glioma_failures": 13
    },
    {
        "model": "ResNet50 U-Net",
        "mean_dice": 0.8570,
        "glioma_mean_dice": 0.7595,
        "severe_glioma_failures": 10
    }
])

display(model_summary.round(4))

summary_path = (
    Path("/content/drive/MyDrive/brain-tumor-mri-segmentation")
    / "results"
    / "segmentation_model_comparison.csv"
)

model_summary.to_csv(summary_path, index=False)

print("\nComparison saved to:", summary_path)

,model,mean_dice,glioma_mean_dice,severe_glioma_failures
0,Baseline U-Net,0.8353,0.6830,22
1,Augmented U-Net,0.8417,0.7004,17
2,U-Net + Tversky,0.8510,0.7352,13
3,ResNet50 U-Net,0.8570,0.7595,10



Comparison saved to: /content/drive/MyDrive/brain-tumor-mri-segmentation/results/segmentation_model_comparison.csv


In [ ]:
# Build test dataset using the existing preprocessing pipeline
test_ds = build_dataset(test_df, training=False)

# Apply the same batching configuration as validation
test_ds = test_ds.batch(16).prefetch(tf.data.AUTOTUNE)

# Verify the dataset
print("Test images:", len(test_df))
print("Expected batches:", (len(test_df) + 15) // 16)
print("Element specification:", test_ds.element_spec)

for images, masks in test_ds.take(1):
    print("\nFirst batch:")
    print("Images:", images.shape, images.dtype)
    print("Masks:", masks.shape, masks.dtype)
    print("Image range:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))
    print("Mask range:", float(tf.reduce_min(masks)), float(tf.reduce_max(masks)))

Test images: 853
Expected batches: 54
Element specification: (TensorSpec(shape=(None, 256, 256, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 256, 256, 1), dtype=tf.float32, name=None))

First batch:
Images: (16, 256, 256, 1) <dtype: 'float32'>
Masks: (16, 256, 256, 1) <dtype: 'float32'>
Image range: 0.0 0.9745097756385803
Mask range: 0.0 1.0


In [ ]:
from pathlib import Path
import tensorflow as tf

checkpoint_dir = Path(
    "/content/drive/MyDrive/brain-tumor-mri-segmentation/checkpoints"
)

checkpoint_files = {
    "Baseline U-Net": "unet_baseline_best.keras",
    "Augmented U-Net": "unet_augmented_best.keras",
    "Tversky U-Net": "unet_tversky_best.keras",
    "ResNet50 U-Net": "resnet50_unet_frozen_best.keras",
}

test_models = {}

for model_name, filename in checkpoint_files.items():
    print(f"\nLoading {model_name}...")

    model = tf.keras.models.load_model(
        checkpoint_dir / filename,
        custom_objects={
            "preprocess_input":
                tf.keras.applications.resnet50.preprocess_input
        },
        compile=False,
        safe_mode=True
    )

    test_models[model_name] = model

    print("Loaded successfully")
    print("Input:", model.input_shape)
    print("Output:", model.output_shape)

print("\nModels loaded:", len(test_models), "/ 4")


Loading Baseline U-Net...
Loaded successfully
Input: (None, 256, 256, 1)
Output: (None, 256, 256, 1)

Loading Augmented U-Net...
Loaded successfully
Input: (None, 256, 256, 1)
Output: (None, 256, 256, 1)

Loading Tversky U-Net...
Loaded successfully
Input: (None, 256, 256, 1)
Output: (None, 256, 256, 1)

Loading ResNet50 U-Net...
Loaded successfully
Input: (None, 256, 256, 1)
Output: (None, 256, 256, 1)

Models loaded: 4 / 4


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

results_dir = Path(
    "/content/drive/MyDrive/brain-tumor-mri-segmentation/results"
)
results_dir.mkdir(parents=True, exist_ok=True)

all_test_results = []
test_summary = []
category_summary = []

for model_name, current_model in test_models.items():
    print(f"\nEvaluating {model_name}...")

    predictions_list = []
    masks_list = []

    for images, masks in test_ds:
        predictions = current_model(images, training=False)

        predictions_list.append(predictions.numpy())
        masks_list.append(masks.numpy())

    predictions = np.concatenate(predictions_list, axis=0)
    true_masks = np.concatenate(masks_list, axis=0)

    assert len(predictions) == len(test_df) == 853

    pred_binary = predictions >= 0.5
    true_binary = true_masks >= 0.5

    axes = (1, 2, 3)

    tp = np.sum(pred_binary & true_binary, axis=axes)
    fp = np.sum(pred_binary & ~true_binary, axis=axes)
    fn = np.sum(~pred_binary & true_binary, axis=axes)

    smooth = 1e-6

    results = test_df.copy().reset_index(drop=True)

    results["dice"] = (
        (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
    )
    results["iou"] = (
        (tp + smooth) / (tp + fp + fn + smooth)
    )
    results["precision"] = (
        (tp + smooth) / (tp + fp + smooth)
    )
    results["recall"] = (
        (tp + smooth) / (tp + fn + smooth)
    )

    results["model"] = model_name
    all_test_results.append(results)

    summary = results[
        ["dice", "iou", "precision", "recall"]
    ].mean().to_dict()

    summary["model"] = model_name
    test_summary.append(summary)

    for category, group in results.groupby("tumor_label"):
        category_summary.append({
            "model": model_name,
            "category": category,
            "images": len(group),
            "dice": group["dice"].mean(),
            "iou": group["iou"].mean(),
            "precision": group["precision"].mean(),
            "recall": group["recall"].mean(),
            "severe_failures": int((group["dice"] < 0.2).sum())
        })

    filename = (
        model_name.lower()
        .replace(" ", "_")
        .replace("-", "_")
        + "_test_results.csv"
    )

    results.to_csv(results_dir / filename, index=False)

    print(
        "Mean Dice:",
        round(results["dice"].mean(), 4)
    )

# Save combined results
all_test_results = pd.concat(
    all_test_results,
    ignore_index=True
)

test_summary_df = pd.DataFrame(test_summary)
category_summary_df = pd.DataFrame(category_summary)

test_summary_df.to_csv(
    results_dir / "test_model_comparison.csv",
    index=False
)

category_summary_df.to_csv(
    results_dir / "test_category_comparison.csv",
    index=False
)

print("\n=== OVERALL TEST COMPARISON ===")
print(
    test_summary_df[
        ["model", "dice", "iou", "precision", "recall"]
    ].round(4).to_string(index=False)
)

print("\n=== TEST DICE BY CATEGORY ===")
print(
    category_summary_df.pivot(
        index="model",
        columns="category",
        values="dice"
    ).round(4).to_string()
)

print("\n=== SEVERE GLIOMA FAILURES ===")
print(
    category_summary_df[
        category_summary_df["category"] == "glioma"
    ][["model", "images", "severe_failures"]]
    .to_string(index=False)
)

print("\nResults saved in:", results_dir)


Evaluating Baseline U-Net...
Mean Dice: 0.8209

Evaluating Augmented U-Net...
Mean Dice: 0.8124

Evaluating Tversky U-Net...
Mean Dice: 0.8269

Evaluating ResNet50 U-Net...
Mean Dice: 0.8373

=== OVERALL TEST COMPARISON ===
          model   dice    iou  precision  recall
 Baseline U-Net 0.8209 0.7412     0.8580  0.8452
Augmented U-Net 0.8124 0.7309     0.8374  0.8439
  Tversky U-Net 0.8269 0.7422     0.8052  0.8982
 ResNet50 U-Net 0.8373 0.7556     0.8273  0.9042

=== TEST DICE BY CATEGORY ===
category         glioma  meningioma  pituitary
model                                         
Augmented U-Net  0.6693      0.9154     0.8295
Baseline U-Net   0.6698      0.9256     0.8432
ResNet50 U-Net   0.7364      0.9233     0.8356
Tversky U-Net    0.7117      0.9191     0.8312

=== SEVERE GLIOMA FAILURES ===
          model  images  severe_failures
 Baseline U-Net     254               30
Augmented U-Net     254               31
  Tversky U-Net     254               17
 ResNet50 U-Net     2